In [ ]:
!pip install pytesseract

In [5]:
import cv2
import pytesseract
import numpy as np
import json

def extract_schedule(img_path):
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Adaptive threshold로 테두리 강조
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                   cv2.THRESH_BINARY_INV, 25, 10)

    # 수평/수직 선 검출
    height, width = gray.shape
    kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (width // 50, 1))
    kernel_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, height // 40))
    h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_h, iterations=2)
    v_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_v, iterations=2)
    table_mask = cv2.add(h_lines, v_lines)

    # 테이블 영역 제외하고 내부 OCR
    cell_mask = cv2.subtract(thresh, table_mask)

    contours, _ = cv2.findContours(table_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    cells = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if 40 < w < width / 5 and 20 < h < height / 3:
            cells.append((x, y, w, h))

    # y좌표 클러스터링 (행 구분)
    cells = sorted(cells, key=lambda b: (b[1], b[0]))
    ys = [y for (_, y, _, _) in cells]
    unique_y = []
    for y in ys:
        if not unique_y or abs(unique_y[-1] - y) > 15:
            unique_y.append(y)

    rows = []
    for y in unique_y:
        row_cells = [(x, yy, w, h) for (x, yy, w, h) in cells if abs(yy - y) < 15]
        row_cells.sort(key=lambda b: b[0])
        rows.append(row_cells)

    # OCR
    ocr_config = "--psm 10 -c tessedit_char_whitelist=DN-E"
    data = {}

    for row_idx, row in enumerate(rows[1:]):  # 첫 행은 날짜 헤더 제외
        day_dict = {}
        for col_idx, (x, y, w, h) in enumerate(row[1:31], start=1):  # 30일 기준
            crop = gray[y+5:y+h-5, x+5:x+w-5]
            text = pytesseract.image_to_string(crop, config=ocr_config).strip()
            # 후처리
            text = text.replace(" ", "").replace("-", "")
            if len(text) > 1:  # OCR이 EE, NN, DD처럼 인식되면 첫 문자만
                text = text[0]
            if text not in ["D", "N", "E"]:
                text = "-"
            day_dict[str(col_idx)] = text
        data[str(row_idx + 1)] = day_dict

    return data


result = extract_schedule("test_2.png")
print(json.dumps(result, ensure_ascii=False, indent=2))


{
  "1": {
    "1": "-",
    "2": "N",
    "3": "N",
    "4": "E",
    "5": "-",
    "6": "D",
    "7": "D",
    "8": "E",
    "9": "-",
    "10": "N",
    "11": "N",
    "12": "E",
    "13": "-",
    "14": "D",
    "15": "D",
    "16": "E",
    "17": "-",
    "18": "N",
    "19": "N",
    "20": "E",
    "21": "-",
    "22": "D",
    "23": "D",
    "24": "E",
    "25": "-",
    "26": "N",
    "27": "N",
    "28": "E",
    "29": "-"
  },
  "2": {
    "1": "E",
    "2": "-",
    "3": "D",
    "4": "D",
    "5": "E",
    "6": "-",
    "7": "N",
    "8": "N",
    "9": "E",
    "10": "-",
    "11": "D",
    "12": "D",
    "13": "E",
    "14": "-",
    "15": "N",
    "16": "N",
    "17": "E",
    "18": "-",
    "19": "D",
    "20": "D",
    "21": "E",
    "22": "-",
    "23": "N",
    "24": "N",
    "25": "E",
    "26": "-",
    "27": "D",
    "28": "D",
    "29": "E"
  },
  "3": {
    "1": "D",
    "2": "E",
    "3": "-",
    "4": "N",
    "5": "N",
    "6": "E",
    "7": "-",
    "8": "D",